In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

# Docking into a crystal-ligand pocket

When a structure has a co-crystallized ligand, the binding site is already known.
This notebook docks into that site explicitly:

1. Prepare the protein with Protein Prep and `find_pockets="from-crystal-ligand"`.
2. Get the prepared `Protein` and the crystal `Pocket` with `get_results()` and `get_pockets()`.
3. Inspect the pocket and its docking box.
4. Pass the pocket to `Docking`.

`Docking` always takes an explicit `pocket`. For targets without a known site, use
`find_pockets="novel"` (or `PocketFinder`) to discover candidate pockets instead.

## Setup

In [ ]:
from deeporigin.drug_discovery import (
    Docking,
    Ligand,
    Protein,
    ProteinPrep,
    chemistry,
)

In [ ]:
from deeporigin import projects

projects.create("python-client-dev")

## Load a holo structure

PDB [1EBY](https://www.rcsb.org/structure/1EBY) is HIV-1 protease bound to the
inhibitor `BEB`. Register the structure as-is; Protein Prep handles the ligand,
waters, and protonation.

In [ ]:
protein = Protein.from_pdb_id("1EBY")
protein.sync()
protein

## Prepare the protein and request the crystal pocket

`find_pockets="from-crystal-ligand"` asks Protein Prep to build one pocket from a
crystal ligand. `component_id` names that ligand by its Selection id. `recommend()`
marks `BEB` as `extract`, so it is removed from the prepared receptor and its pose
is kept as a crystal pose.

In [ ]:
prep = ProteinPrep(
    protein,
    find_pockets="from-crystal-ligand",
    component_id="ligand:B:BEB:501:",
)
recommendation = prep.recommend()
recommendation[recommendation["kind"] != "water"][["id", "kind", "label", "decision"]]

In [ ]:
prep

`run()` blocks until preparation completes and returns the prepared `Protein`.

In [ ]:
prepared = prep.run()
prepared

## Get the crystal pocket

`get_pockets()` returns the pockets from the same run. The crystal pocket is
attached to the prepared protein and records which ligand it came from.

In [ ]:
pockets = prep.get_pockets()
pocket = pockets[0]
pocket

In [ ]:
pocket.origin, pocket.component_id, pocket.protein.id == prepared.id

## Inspect the pocket

Show the pocket on the prepared protein:

In [ ]:
pocket.show()

Preview the docking search box. The box is oriented to the crystal ligand:

In [ ]:
pocket.show_box()

## Load the crystal ligand and pose

The extracted ligand is registered on the platform; `pocket.ligand_id` points to
it. `get_crystal_poses()` returns its crystal coordinates, which we use as a
reference below.

In [ ]:
crystal_ligand = Ligand.from_id(pocket.ligand_id)
crystal_pose = prep.get_crystal_poses().poses[0]
crystal_pose.download()
crystal_ligand

## Dock into the crystal pocket

Pass the prepared protein and the crystal pocket to `Docking` explicitly. Here we
re-dock the crystal ligand, which lets us check the result against the crystal pose.

In [ ]:
docking = Docking(protein=prepared, pocket=pocket, ligand=crystal_ligand)
docking

In [ ]:
docking.show_box()

In [ ]:
poses = docking.run()
poses.to_dataframe()[["pose_score", "binding_energy", "best_pose"]]

## Compare with the crystal pose

RMSD (Å) between the best-scoring docked pose and the crystal pose, without realignment:

In [ ]:
poses.download()
best = max(
    poses.poses,
    key=lambda pose: float("-inf") if pose.pose_score is None else pose.pose_score,
)
chemistry.pose_rmsd(best.mol, crystal_pose.mol)

In [ ]:
prepared.show(poses=poses)

## Next steps

- Dock other ligands into the same `pocket`, or use `ConstrainedDocking` with
  `reference_pose=crystal_pose.to_ligand()` to keep analogs close to the crystal pose.
- For a target without a known site, set `find_pockets="novel"` and call
  `prep.start()` instead of `run()`, then choose a pocket from `prep.get_pockets()`.